# Beautiful Soup Tutorial

In this tutorial, we will learn how to:

1. Use the requests library to fetch the HTML content of a webpage and then use Beautiful Soup to parse it.
2. Extract specific elements and data from the parsed HTML.
3. Use requests to send HTTP requests via an API and handle the response.
4. Use the json library to parse JSON data from an API response.

**beautifulsoup** is a Python library for parsing HTML and XML documents. It creates a parse tree for parsed pages that can be used to extract data from HTML, which is useful for web scraping. [beautifulsoup documentation](https://pypi.org/project/beautifulsoup4/)

To pip install beautifulsoup4, run the following command in your terminal:

```bash
pip install beautifulsoup4
```

**requests** is a simple and elegant HTTP library for Python, built for human beings. It allows you to send HTTP requests easily and access the response data in various formats. [requests documentation](https://docs.python-requests.org/en/latest/)

To pip install requests, run the following command in your terminal:

```bash
pip install requests
```

**json** is a built-in Python library for parsing JSON data. It allows you to convert JSON strings into Python objects and vice versa. [json documentation](https://docs.python.org/3/library/json.html)

The json module is included in Python’s standard library and is preinstalled with all standard Python distributions. You do not need to install it separately—just import it with `import json`.

In [ ]:
# import beautifulsoup4

import requests
from bs4 import BeautifulSoup
import json


## Status Codes

When you make a request to a webpage using the requests library, you receive a response object. This response object contains a status code that indicates the result of the request. A status code of 200 means that the request was successful, while a status code of 404 means that the page was not found.

Common status codes include:
- 200: OK
- 301: Moved Permanently
- 302: Found (Temporary Redirect)
- 400: Bad Request
- 401: Unauthorized
- 403: Forbidden
- 404: Not Found
- 500: Internal Server Error
- 503: Service Unavailable
- 504: Gateway Timeout
- 505: HTTP Version Not Supported

In [ ]:
# use requests to get the status code of the page

url = "https://www.binghamton.edu/history/"
response = requests.get(url)
print(f"Status Code: {response.status_code}")

In [ ]:
# Get the HTML content of the page
html_content = response.text
print(html_content)

The requests library only fetches the static HTML returned by the server, not content rendered or injected by JavaScript in the browser.

To scrape dynamically loaded content, use a tool like Selenium or Playwright, which automates a real browser and can execute JavaScript. [Selenium documentation](https://www.selenium.dev/documentation/)

To process again with Selenium, you would need to install it and a web driver (like ChromeDriver for Google Chrome). Here’s a basic example of how to use Selenium to fetch a webpage:

```python
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# Set up Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode (without opening a browser window)
service = Service('/path/to/chromedriver')  # Update with the path to your ChromeDriver
driver = webdriver.Chrome(service=service, options=chrome_options)
driver.get('https://www.example.com')
html = driver.page_source  # Get the rendered HTML
driver.quit()  # Close the browser
```

We would need to pip install:

```bash
pip install selenium
```

Installation and usage of Selenium is more complex than requests and Beautiful Soup. For this tutorial, we will focus on using requests and Beautiful Soup to scrape static content from webpages.

## Parsing with Beautiful Soup

Now that we have the HTML content of the page, we can use Beautiful Soup to parse it.

Let's get the title of the page from the header of the HTML document. The title is usually found within the `<title>` tag in the `<head>` section of the HTML.

In [ ]:
# Get the title of the page
soup = BeautifulSoup(html_content, 'html.parser')
title = soup.title.string
print(f"Title: {title}")
print(f"Meta Description: {soup.find('meta', attrs={'name': 'description'})['content']}")

Now, let's extract links from the body of the page. We can use the `find_all` method to find all the `<a>` tags, which represent links in HTML. 

A link in HTML is represented by an `<a>` tag, which stands for "anchor". The `<a>` tag typically has an `href` attribute that contains the URL of the link. For example:

```html
<a href="https://www.example.com">Example Link</a>
```

We can then extract the `href` attribute from each `<a>` tag to get the URL of the link:

In [ ]:
# Import links from https://www.binghamton.edu/history/
url = "https://www.binghamton.edu/history/"
response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')
links = soup.find_all('a')

# Now we loop through the links and print the href attribute of each link, which contains the URL of the link.
for link in links:
    print(link.get('href'))


If we want to extract the links and the text of the link (between the `<a>` and `</a>` tags), and then format them as markdown links, we can do that as well:

In [ ]:
# Now extract not just the links, but also the text of the links
for link in links:
    href = link.get('href')
    text = link.text
    print(f"Link: [{href}]({text})")

A basic element of html is the `p` tag, which stands for "paragraph". The `p` tag is used to define a paragraph of text in an HTML document. For example:

```html
<p>This is a paragraph of text.</p>
```

Another element is the h1, h2, h3, h4, h5, and h6 tags, which are used to define headings in an HTML document. The number after the "h" indicates the level of the heading, with h1 being the highest level and h6 being the lowest. For example:

```html
<h1>This is a heading level 1</h1>
<h2>This is a heading level 2</h2>
<!-- and so on... -->
```

Extract the headings and the paragraphs:

In [ ]:
# Extract the headings and paragraphs from /graduate page
# Ensure that the text remains in order with each heading followed by the paragraphs that follow it until the next heading.
# We'll iterate through the children of the main content area, grouping paragraphs under their preceding heading.

gradurl = url + "graduate/"
response = requests.get(gradurl)
soup = BeautifulSoup(response.content, 'html.parser')

# Find the main content area (adjust selector as needed)
main_content = soup.body  # You may want to use a more specific selector if available, e.g., soup.find('div', class_='main-content')

# Initialize variables to keep track of the current heading and its associated paragraphs
current_heading = None
content = []

# Iterate through the children of the main content area and group paragraphs under their preceding heading
for elem in main_content.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'], recursive=True):
    if elem.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
        current_heading = {'heading': f"{elem.name}: {elem.get_text(strip=True)}", 'paragraphs': []}
        content.append(current_heading)
    elif elem.name == 'p' and current_heading is not None:
        current_heading['paragraphs'].append(elem.get_text(strip=True))

# Print the structured content
for section in content:
    print(section['heading'])
    for para in section['paragraphs']:
        print(f"    {para}")

Sometimes we want to extract the images on a webpage. Images in HTML are represented by the `<img>` tag, which has a `src` attribute that contains the URL of the image. For example:

```html
<img src="https://www.example.com/image.jpg" alt="Example Image">
```

Let's get all of the images from the base url and put them in a folder called "history_images".

In [ ]:
# Get html images from the base url and save them to a folder called "history_images"
import os
from urllib.parse import urljoin
import re

url = "https://www.binghamton.edu/history/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# 1. Extract <img> tag images
img_urls = set()
for img in soup.find_all('img'):
    img_url = img.get('src') or img.get('data-src') or img.get('data-lazy')
    if img_url:
        full_img_url = urljoin(url, img_url)
        img_urls.add(full_img_url)

# 2. Extract background-image URLs from style attributes
for tag in soup.find_all(style=True):
    style = tag['style']
    matches = re.findall(r'background-image:\s*url\(([^)]+)\)', style)
    for match in matches:
        # Remove quotes if present
        img_url = match.strip('"\' )')
        full_img_url = urljoin(url, img_url)
        img_urls.add(full_img_url)

# 3. Save all found images
os.makedirs('history_images', exist_ok=True)
for full_img_url in img_urls:
    img_name = os.path.basename(full_img_url)
    img_path = os.path.join('history_images', img_name)
    try:
        img_response = requests.get(full_img_url)
        if img_response.status_code == 200:
            with open(img_path, 'wb') as f:
                f.write(img_response.content)
            print(f"Saved image: {img_path}")
        else:
            print(f"Failed to download image: {full_img_url} (Status code: {img_response.status_code})")
    except Exception as e:
        print(f"Error downloading image: {full_img_url} - {e}")

The other images on the page are loaded dynamically with JavaScript, so we won't be able to extract them with Beautiful Soup. To extract those images, we would need to use a tool like Selenium or Playwright that can execute JavaScript and render the page as a browser would.

In what follows, we will use requests, os, beautifulsoup, and json in a real-world example of scraping data from a primary document called the Florentine Codex.

## Digital Florentine Codex

The [Digital Florentine Codex](https://florentinecodex.getty.edu/) is a project that has digitized the Florentine Codex, a 16th-century ethnographic research study in Mesoamerica by the Spanish Franciscan friar Bernardino de Sahagún. The codex is a valuable resource for understanding the culture, history, and society of the Aztec people. The digital version allows researchers and the public to access and explore this important historical document online.

The 16th-century manuscript’s artists painted about 2,400 scenes and decorative elements (2,472 in total). This total number includes 1,844 images with narrative content and 628 decorative elements or grotesques distributed irregularly throughout the books. For example, Book 5 contains only 9 images and 3 decorative elements, while Book 11 contains a total of 1,137 “pictorial statements.”

The URL for Book 5 is [https://florentinecodex.getty.edu/book/5](https://florentinecodex.getty.edu/book/5).

In what follows, we will scrape the content of Book 5 of the Digital Florentine Codex and extract the images.

In [ ]:
# Fetch the images from book 5

import requests
import os
import json
from bs4 import BeautifulSoup

# Create a directory to save the images
output_dir = "book5_images"
os.makedirs(output_dir, exist_ok=True)

# Function to get the image URL for a given folio
def get_folio_image_url(book_num, folio):
    url = f"https://florentinecodex.getty.edu/book/{book_num}/folio/{folio}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    script = soup.find('script', id='__NEXT_DATA__')
    if not script:
        print(f"No data for folio {folio}")
        return None
    data = json.loads(script.string)
    files = data.get('props', {}).get('pageProps', {}).get('data', {}).get('files', {})
    return files.get('folio_jpg')

# Example: Download folios 1r to 10v
folios = []
for n in range(1, 11):
    folios.append(f"{n}r")
    folios.append(f"{n}v")

for folio in folios:
    img_url = get_folio_image_url(5, folio)
    if img_url:
        img_path = os.path.join(output_dir, f"book5_{folio}.jpg")
        print(f"Downloading {img_url} -> {img_path}")
        img_data = requests.get(img_url).content
        with open(img_path, "wb") as f:
            f.write(img_data)
    else:
        print(f"Image not found for folio {folio}")

While the Florentine Codex contains original texts in Nahuatl and Spanish, the Digital Florentine Codex provides English translations of each of these texts, allowing researchers to compare the differences between how the FC presented information to Nahuatl- and Spanish-speaking audiences.

Let's extract the Nahuatl-to-English translations from Book 5 of the Digital Florentine Codex, ff. 1 - 10.

Let's look at the html for one of the folios, for example, f. 1r: [https://florentinecodex.getty.edu/book/5/folio/1r](https://florentinecodex.getty.edu/book/5/folio/1r).

Let's fetch the HTML for that page and see how the Nahuatl text and its English translation are stored in the HTML.

The Digital Florentine Codex site embeds all the Nahuatl and English translation text as JSON data inside a `script` tag with id `__NEXT_DATA__` in the initial HTML response:

```html
<script id="__NEXT_DATA__">
```

This tag is present even when using requests, so you can extract and parse the data directly with BeautifulSoup and json—no JavaScript execution is needed.

In [ ]:
# Extract the nahuatl text and the nahuatl-to-english translations for book 5 ff 1-10

import requests
import os
import json
from bs4 import BeautifulSoup

# Create a directory to save the text files
output_dir = "book5_nahuatl"
os.makedirs(output_dir, exist_ok=True)

# Function to get the Nahuatl text for a given folio
def get_nahuatl_text(book_num, folio):
    url = f"https://florentinecodex.getty.edu/book/{book_num}/folio/{folio}"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    # The Nahuatl text and its English translation are stored in the __NEXT_DATA__ script tag as JSON data
    script = soup.find('script', id='__NEXT_DATA__')
    if not script:
        print(f"No data for folio {folio}")
        return None
    data = json.loads(script.string)
    texts = data.get('props', {}).get('pageProps', {}).get('data', {}).get('texts', {})
    nahuatl = texts.get('nahuatl_col', [])
    return nahuatl

# Example: Extract Nahuatl text for folios 1r to 10v
folios = []
for n in range(1, 11):
    folios.append(f"{n}r")
    folios.append(f"{n}v")

# Loop through the folios and save the Nahuatl text to separate files
for folio in folios:
    nahuatl = get_nahuatl_text(5, folio)
    if nahuatl:
        outpath = os.path.join(output_dir, f"book5_{folio}_nahuatl.md")
        with open(outpath, "w", encoding="utf-8") as f:
            for entry in nahuatl:
                f.write(entry.get("markdown", "") + "\n\n")
        print(f"Saved Nahuatl text for folio {folio}")
    else:
        print(f"Nahuatl text not found for folio {folio}")

## API Usage

An API is 


### Ecology of Crisis API

The Ecology of Crisis database (https://ecocrisis.net) contains historical data about famines, epidemics, climate events, and other crises from around the world. It has a much simpler API structure that's great for learning.

Key features:
- Public endpoints (no authentication needed for reading data)
- Historical events categorized by themes (famine, epidemic, flood, etc.)
- Geographic and temporal data
- Time series analysis

Let's use the requests library in Python to interact with the API and process the data programmatically. This allows us to automate data retrieval, handle responses more effectively, and integrate the API data into our analysis pipelines.

In [ ]:
# Get public statistics from the Ecology of Crisis database
import requests
import json

base_url = "https://ecocrisis.net"

# Get overall statistics
response = requests.get(f"{base_url}/eventinfo/public/stats")

stats = response.json()

print("Database Statistics:")
print(json.dumps(stats, indent=2))

Another endpoint is `event/{id}` which will return detailed information about a specific event. For example, `event/101` will return details about the first event in the database.

In [ ]:
import requests

event_id = 101

event_resource = "/eventinfo/public/event"

url = f"{base_url}{event_resource}/{event_id}"

response = requests.get(url)

print(response.json())


We could write the json to a file:

In [ ]:
# Write json data to a file called event_99.json but use event_id in the filename instead of 99
with open(f"event_{event_id}.json", "w", encoding="utf-8") as f:
    json.dump(event, f, ensure_ascii=False, indent=2)

We could also get a more readable output:

In [ ]:
print(f"\nEvent ID: {event.get('id')}")
print(f"Summary: {event.get('summary')}")
print(f"Start Date: {event.get('start_year')}")
print(f"Quote: {event.get('event_quote')}")
print(f"Sources: {event.get('event_source_info')}")

# print all the themes associated with the event, or "None" if there are no themes. Themes are classified into classes, and classes into domains, so add that in parentheses after the theme.
print("Themes:")
themes = event.get('themes', [])
if themes:
    for theme in themes:
        theme_name = theme.get('name', 'Unknown')
        theme_class = theme.get('eventClass', {}).get('name', 'Unknown')
        theme_domain = theme.get('eventClass', {}).get('domain', {}).get('name', 'Unknown')
        print(f"  - {theme_name} ({theme_class}, {theme_domain})")
else:
    print("  - None")

# print all the locations associated with the event, or "None" if there are no locations
print("Locations:")
for loc in event.get('eventGeoLocationSet', []):
    loc_desc = []
    if loc.get('site'):
        loc_desc.append(f"Site: {loc['site']}")
    if loc.get('municipality'):
        loc_desc.append(f"Municipality: {loc['municipality']}")
    if loc.get('state'):
        loc_desc.append(f"State: {loc['state']}")
    if loc.get('country'):
        loc_desc.append(f"Country: {loc['country']}")
    if loc.get('lat') and loc.get('lon'):
        loc_desc.append(f"Coords: ({loc['lat']}, {loc['lon']})")
    print("    - " + ", ".join(loc_desc) if loc_desc else "    - Unknown")

## Query Specific Year

A different endpoint allows us to query events by year or a range of years.

The endpoint is `/eventinfo/public/events/by-year?startYear=YYYY[&endYear=YYYY]`

Let's query all events from a specific year (1697) to see what crisis events occurred that year.

The equivalent curl command is:
```bash
curl -X GET "https://ecocrisis.net/eventinfo/public/events/by-year?startYear=1697&endYear=1697"
```

Let's begin by fetching the first event from the year 1697 using the requests library in Python.

In [ ]:
# Get events for 1697 as json file (first event only)
import requests

base_url = "https://ecocrisis.net"

year_range_resource = "/eventinfo/public/events/by-year"

# We can use params to specify the start and end year for our query. In this case, we want to query events that occurred in the year 1697, so we set both startYear and endYear to 1697.

params = {
    "startYear": 1697,
    "endYear": 1697
}

response = requests.get(f"{base_url}{year_range_resource}", params=params)
events_1697 = response.json()

# Print only the first event in a readable format
print(json.dumps(events_1697[0], indent=2, ensure_ascii=False))

Often, we will want to save the json output to a file for later analysis. We can do this using the `json` library in Python to write the response data to a file.

In [ ]:
import json
with open("events_1697.json", "w", encoding="utf-8") as f:
    json.dump(events_1697[:20], f, ensure_ascii=False, indent=2)

Let's fetch again but print out the year and summary information for each event.

In [ ]:
# Fetch all events from the API and print only year, quote, and source for those with start_year == 1697

# Loop through the filtered events and print the year, quote, and source for each
for event in events_1697:
    print("Year:", event.get('start_year', 'Unknown'))
    print("Quote:", event.get('event_quote', 'No quote'))
    print("Source:", event.get('event_source_info', 'Unknown'))
    # the * 40 will print a line of 40 dashes to separate each event in the output
    print("-" * 40)